In [ ]:
import cv2
import numpy as np

def augment_image(image_path, save_dir):
    """
    Function: Implement multi-dimensional data augmentation based on ResNet paper's preprocessing specifications
    Core Logic: Refer to the augmentation strategy in the paper [5-60] to avoid feature distortion caused by excessive augmentation
    """
    # Read image and convert to RGB (OpenCV defaults to BGR, adapting to subsequent visualization and model input)
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_name = image_path.split("/")[-1].split(".")[0]
    
    # 1. Rotate 15° (the paper does not specify a specific angle; ±15° is chosen to balance diversity and feature retention)
    h, w = img_rgb.shape[:2]
    M_rot = cv2.getRotationMatrix2D((w//2, h//2), 15, 1)
    img_rot = cv2.warpAffine(img_rgb, M_rot, (w, h))
    
    # 2. Horizontal flipping (core augmentation method in ResNet paper's ImageNet experiments, forcing key feature retention)
    img_flip = cv2.flip(img_rgb, 1)
    
    # 3. Brightness adjustment (±20%, referring to the paper's "moderate augmentation" principle to avoid skin texture distortion)
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_hsv[:, :, 2] = np.clip(img_hsv[:, :, 2] * 1.2, 0, 255)  # Brightness +20%
    img_bright = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)
    
    # Uniformly scale to 224×224 (ResNet paper's input size, adapting to the MBConv module of EfficientNet)
    aug_imgs = [img_rot, img_flip, img_bright]
    aug_names = ["_rot15", "_flip", "_bright120"]
    for idx, (aug_img, aug_name) in enumerate(zip(aug_imgs, aug_names)):
        img_resized = cv2.resize(aug_img, (224, 224))
        save_path = f"{save_dir}/{img_name}{aug_name}.jpg"
        cv2.imwrite(save_path, cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR))
    
    return f"Successfully generated 3 augmented samples for {img_name}, complying with ResNet preprocessing specifications"

# Call the function (taking FaceForensics++ real samples as an example)
result = augment_image(
    image_path="data/raw/real/001.jpg",
    save_dir="data/processed/augmented/real"
)
print(result)

In [ ]:
def crop_face_from_video(video_path, save_dir, frame_interval=10):
    """
    Function: Extract frames from Deepfake videos and crop faces, focusing on key detection regions
    Core Logic: Draw on the padding strategy in ResNet paper [5-111] to ensure the integrity of face features
    """
    # Load OpenCV pre-trained Haar cascade classifier (lightweight, adapting to laptop CPU)
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    face_count = 0
    video_name = video_path.split("/")[-1].split(".")[0]
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Extract 1 frame every 10 frames (balancing data volume and processing efficiency, adapting to lightweight requirements)
        if frame_count % frame_interval == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
            
            # Only retain frames with a single face (avoiding multi-face interference, focusing on Deepfake's core forgery region)
            if len(faces) == 1:
                x, y, w, h = faces[0]
                # Padding strategy: Expand the region by 10%, referring to the ResNet paper's principle of "completely retaining features"
                pad = int(min(w, h) * 0.1)
                x_pad = max(0, x - pad)
                y_pad = max(0, y - pad)
                w_pad = min(frame_rgb.shape[1] - x_pad, w + 2*pad)
                h_pad = min(frame_rgb.shape[0] - y_pad, h + 2*pad)
                face_img = frame_rgb[y_pad:y_pad+h_pad, x_pad:x_pad+w_pad]
                
                # Uniformly scale to 224×224, consistent with data augmentation output
                face_resized = cv2.resize(face_img, (224, 224))
                save_path = f"{save_dir}/{video_name}_face{face_count}.jpg"
                cv2.imwrite(save_path, cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
                face_count += 1
        
        frame_count += 1
    
    cap.release()
    return f"Video {video_name} processed successfully, generating {face_count} standardized face images"

# Call the function (taking FaceForensics++ fake videos as an example)
crop_result = crop_face_from_video(
    video_path="data/raw/fake/001.mp4",
    save_dir="data/processed/faces/fake"
)
print(crop_result)

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

def create_dataset_label(root_dir, save_csv_path):
    """
    Function: Generate standardized label tables and divide training/test sets
    Core Logic: Refer to the dataset division specifications in ResNet paper [5-64] to ensure label balance
    """
    # Collect data paths and labels (0=real, 1=fake, meeting the needs of binary classification tasks)
    data = []
    for label, folder in enumerate(["real", "fake"]):
        folder_path = os.path.join(root_dir, folder)
        for img_name in os.listdir(folder_path):
            if img_name.endswith(".jpg"):
                img_path = os.path.join(folder_path, img_name)
                # Record key information, referring to the ResNet paper's "data traceability" principle
                data.append({
                    "image_path": img_path,
                    "label": label,
                    "data_source": "FaceForensics++",
                    "image_size": "224×224",
                    "augmentation_type": img_name.split("_")[-1].split(".")[0]  # Mark augmentation type
                })
    
    # Generate total label table for subsequent tracing and management
    df = pd.DataFrame(data)
    df.to_csv(save_csv_path, index=False, encoding="utf-8")
    total_real = len(df[df["label"] == 0])
    total_fake = len(df[df["label"] == 1])
    print(f"Total label table generated: {len(df)} data entries (real: {total_real} entries, fake: {total_fake} entries)")
    
    # Divide into training/test sets at 8:2 ratio, stratify parameter ensures balanced label distribution (referring to ResNet paper's balance principle)
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["label"]
    )
    train_df.to_csv("labels/train_labels.csv", index=False, encoding="utf-8")
    test_df.to_csv("labels/test_labels.csv", index=False, encoding="utf-8")
    
    # Verify label distribution
    train_real = len(train_df[train_df["label"] == 0])
    train_fake = len(train_df[train_df["label"] == 1])
    test_real = len(test_df[test_df["label"] == 0])
    test_fake = len(test_df[test_df["label"] == 1])
    print(f"Training set: {len(train_df)} entries (real: {train_real} entries, fake: {train_fake} entries)")
    print(f"Test set: {len(test_df)} entries (real: {test_real} entries, fake: {test_fake} entries)")
    
    return "Label table generation and division completed, complying with ResNet dataset management specifications"

# Call the function (root directory contains real/fake subfolders)
label_result = create_dataset_label(
    root_dir="data/processed",
    save_csv_path="labels/total_labels.csv"
)
print(label_result)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def tensor_basics_demo():
    """
    Function: Demonstrate core TensorFlow operations, adapting to CNN input requirements
    Core Logic: Refer to the feature map operation logic in ResNet paper [5-42]
    """
    # Simulate preprocessed image tensor (batch_size=2, height=224, width=224, channels=3)
    img_tensor = tf.random.normal(shape=(2, 224, 224, 3))
    print("1. Shape of simulated image tensor (adapting to CNN input):", img_tensor.shape)  # Output: (2, 224, 224, 3)
    
    # Pooling operation (used in ResNet paper to halve feature map size while retaining key features)
    pooled_tensor = layers.MaxPooling2D(pool_size=(2, 2))(img_tensor)
    print("2. Shape of pooled tensor (size halved):", pooled_tensor.shape)  # Output: (2, 112, 112, 3)
    
    return img_tensor, pooled_tensor

def simple_residual_block(input_tensor, filters=32):
    """
    Function: Implement a simplified residual block, drawing on the y=F(x)+x logic in ResNet paper [5-43]
    Core Structure: Convolution → BN → ReLU → Convolution → BN → Shortcut Addition → ReLU
    """
    # Residual mapping F(x): Refer to the ResNet paper's layer order design of "convolution → BN → activation"
    x = layers.Conv2D(filters, (3, 3), padding="same")(input_tensor)
    x = layers.BatchNormalization()(x)  # BN layer follows convolution closely to improve convergence stability (paper [5-60])
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut connection: Direct addition when dimensions are consistent; 1×1 convolution projection when dimensions are inconsistent (paper [5-47])
    if input_tensor.shape[-1] == x.shape[-1]:
        output_tensor = layers.Add()([x, input_tensor])
    else:
        input_proj = layers.Conv2D(filters, (1, 1), padding="same")(input_tensor)
        output_tensor = layers.Add()([x, input_proj])
    
    output_tensor = layers.ReLU()(output_tensor)
    return output_tensor

# Call the function for verification
tensor_basics_demo()
# Test the residual block (input is a 224×224 image tensor)
input_tensor = tf.random.normal(shape=(2, 224, 224, 3))
res_block_output = simple_residual_block(input_tensor, filters=32)
print("3. Shape of residual block output tensor:", res_block_output.shape)  # Output: (2, 224, 224, 32)